# L13c: MCP Tools, Resources, and Trust Boundaries

MCP gives a host a standard way to discover capabilities offered by a server. It does not grant those capabilities authority by itself.

> **Learning objectives**
>
> - Distinguish host, client, server, resource, and tool roles.
> - Read typed capability schemas before calling them.
> - Separate computation tests from protocol tests.
> - Bound side effects and authority with least privilege.


## Inspect the local resource first


In [1]:
from pathlib import Path
import json

week_root = Path("..").resolve()
network_path = week_root / "data" / "urea-cycle-network.json"
network = json.loads(network_path.read_text(encoding="utf-8"))
{
    "network_id": network["network_id"],
    "species": len(network["species"]),
    "reactions": len(network["reactions"]),
}


{'network_id': 'urea-cycle', 'species': 18, 'reactions': 19}

## Roles and capabilities

- **Host:** coordinates the user, model, and one or more MCP clients.
- **Client:** protocol component connecting the host to one server.
- **Server:** exposes a deliberately bounded set of capabilities.
- **Resource:** data a client may read; here, the urea-cycle JSON model.
- **Tool:** a typed computation; here, summarize the network or check `S*v`.

Capability discovery tells the client what is available. The host still decides whether a requested call is appropriate and authorized.


In [2]:
capability_contract = {
    "resource": {
        "uri": "cheme://metabolic-network/urea-cycle",
        "effect": "read one committed JSON model",
    },
    "tools": {
        "summarize_network": {"network_id": "string"},
        "check_flux_balance": {
            "flux": "array[number]",
            "tolerance": "number >= 0",
            "network_id": "string",
        },
    },
}
print(json.dumps(capability_contract, indent=2))


{
  "resource": {
    "uri": "cheme://metabolic-network/urea-cycle",
    "effect": "read one committed JSON model"
  },
  "tools": {
    "summarize_network": {
      "network_id": "string"
    },
    "check_flux_balance": {
      "flux": "array[number]",
      "tolerance": "number >= 0",
      "network_id": "string"
    }
  }
}


## Trust-boundary inventory

The common server is local and read-only. It has no capability to modify network files, change reaction bounds, execute arbitrary commands, inspect secrets, or make external requests. An unsupported tool name must fail closed.


In [3]:
server_source = week_root / "python" / "src" / "cheme5800_mcp" / "server.py"
source = server_source.read_text(encoding="utf-8")
{
    "decorated_tools": source.count("@mcp.tool()"),
    "decorated_resources": source.count("@mcp.resource("),
    "contains_shell_execution": "subprocess" in source,
    "contains_file_write": ".write_text(" in source,
}


{'decorated_tools': 2,
 'decorated_resources': 1,
 'contains_shell_execution': False,
 'contains_file_write': False}

## Protocol-version note

The official Python SDK 2.x implements the `2026-07-28` protocol. That revision uses stateless discovery rather than teaching students to memorize the older session handshake. The course emphasizes the durable abstractions (schemas, discovery, tools, resources, and trust boundaries) and revalidates SDK/Inspector details before release.

Official references: [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk) and [2026-07-28 specification announcement](https://blog.modelcontextprotocol.io/posts/2026-07-28/).


## Computation tests are not protocol tests

A numerical unit test can verify `S*v` without starting an MCP server. A protocol test verifies discovery, serialization, typed invocation, and error propagation. We need both, but they answer different questions.
